# Solución guiada corregida / Corrected guided solution

Este notebook es una **solución guiada** para analizar una imagen FITS de un disco protoplanetario.  
This notebook is a **guided solution** for analyzing a FITS image of a protoplanetary disk.

Incluye carga del FITS, visualización, deproyección geométrica, mapa polar, perfil radial e identificación simple de subestructuras.  
It includes FITS loading, visualization, geometrical deprojection, polar map, radial profile, and simple substructure identification.

**Importante / Important:**  
La medición de brechas aquí es una medición de **contraste de intensidad**, no una medición directa de masa ni de depleción física del material.  
The gap measurement here is an **intensity contrast** measurement, not a direct measurement of mass or physical depletion.


## 1. Importar librerías / Import libraries

Si falta alguna librería, instala primero:  
If any library is missing, install it first:

```bash
pip install numpy matplotlib astropy scipy
```


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy.visualization import ImageNormalize, AsinhStretch, PercentileInterval

from scipy.ndimage import map_coordinates
from scipy.signal import find_peaks


## 2. Elegir el disco / Select the disk

Cambia `DISK_NAME` para analizar **HD 163296** o **IM Lup**.  
Change `DISK_NAME` to analyze **HD 163296** or **IM Lup**.

Los parámetros geométricos son valores aproximados usados para esta actividad.  
The geometrical parameters are approximate values used for this activity.


In [ ]:
# Elige el disco aquí / Choose the disk here
DISK_NAME = "HD163296"   # options/opciones: "HD163296" or "IMLup"

disk_parameters = {
    "HD163296": {
        "file_name": "HD163296_continuum.fits",
        "repo_path": Path("data/actividades/actividad_hd163296/HD163296_continuum.fits"),
        "inclination_deg": 46.7,
        "pa_deg": 133.3,
        "distance_pc": 101.0,
        "crop_size_pixels": 1100,
        "r_max_arcsec": 1.55,
        "selected_gap_arcsec": 1.22,
    },
    "IMLup": {
        "file_name": "IMLup_continuum.fits",
        "repo_path": Path("data/actividades/actividad_im_lup/IMLup_continuum.fits"),
        "inclination_deg": 48.0,
        "pa_deg": 144.0,
        "distance_pc": 161.0,
        "crop_size_pixels": 1500,
        "r_max_arcsec": 2.2,
        "selected_gap_arcsec": 1.0,
    },
}

params = disk_parameters[DISK_NAME]

print("Selected disk / Disco seleccionado:", DISK_NAME)
for key, value in params.items():
    print(f"{key}: {value}")


## 3. Buscar el archivo FITS / Find the FITS file

El código busca el FITS en varias ubicaciones razonables: la carpeta actual, la carpeta del repositorio o el paquete descargado.  
The code searches for the FITS file in several reasonable locations: the current folder, the repository folder, or the downloaded package.


In [ ]:
def find_fits_file(params):
    file_name = params["file_name"]
    repo_path = params["repo_path"]

    candidates = [
        Path(file_name),
        Path.cwd() / file_name,
        repo_path,
        Path.cwd() / repo_path,
        Path.cwd() / "morphology_activity_package" / file_name,
    ]

    # Buscar también en carpetas superiores / also search parent folders
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        candidates.append(parent / file_name)
        candidates.append(parent / repo_path)
        candidates.append(parent / "data" / "actividades" / file_name)

    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        f"No se encontró el FITS / FITS file not found: {file_name}\n"
        "Deja el FITS en la misma carpeta del notebook o ejecuta el notebook desde el repositorio."
    )

fits_path = find_fits_file(params)
print("FITS path / Ruta del FITS:")
print(fits_path)


## 4. Abrir el FITS / Open the FITS

El FITS se abre con `astropy.io.fits`.  
The FITS file is opened with `astropy.io.fits`.

`np.squeeze` elimina dimensiones extra de tamaño 1, frecuentes en imágenes ALMA.  
`np.squeeze` removes extra dimensions of size 1, which are common in ALMA images.


In [ ]:
with fits.open(fits_path) as hdul:
    hdul.info()
    header = hdul[0].header.copy()
    data = hdul[0].data

image = np.squeeze(data).astype(float)
image[~np.isfinite(image)] = np.nan

if image.ndim != 2:
    raise ValueError(f"Expected a 2D image after squeeze, but got {image.ndim} dimensions.")

print("Image shape / Tamaño de la imagen:", image.shape)
print("Minimum / Mínimo:", np.nanmin(image))
print("Maximum / Máximo:", np.nanmax(image))
print("Unit / Unidad:", header.get("BUNIT", "not specified"))


## 5. Escala angular, centro y recorte / Angular scale, center, and crop

Aquí usamos la escala angular del header FITS.  
Here we use the angular scale from the FITS header.

A diferencia de una visualización simple, para la geometría conviene conservar el signo de `CDELT1` y `CDELT2`.  
Unlike a simple visualization, for the geometry it is useful to keep the sign of `CDELT1` and `CDELT2`.


In [ ]:
ny, nx = image.shape

# Escala angular firmada / signed angular scale
cdelt_x = header["CDELT1"] * 3600.0  # arcsec/pixel
cdelt_y = header["CDELT2"] * 3600.0  # arcsec/pixel
pixel_scale = 0.5 * (abs(cdelt_x) + abs(cdelt_y))

x_center = header.get("CRPIX1", nx / 2) - 1
y_center = header.get("CRPIX2", ny / 2) - 1

crop_size = params["crop_size_pixels"]
half = crop_size // 2

x0 = int(max(x_center - half, 0))
x1 = int(min(x_center + half, nx))
y0 = int(max(y_center - half, 0))
y1 = int(min(y_center + half, ny))

image_crop = image[y0:y1, x0:x1]

x_center_crop = x_center - x0
y_center_crop = y_center - y0

print(f"Signed CDELT x / CDELT x firmado: {cdelt_x:.6f} arcsec/pixel")
print(f"Signed CDELT y / CDELT y firmado: {cdelt_y:.6f} arcsec/pixel")
print(f"Pixel scale / Escala de pixel: {pixel_scale:.6f} arcsec/pixel")
print(f"Cropped image shape / Tamaño del recorte: {image_crop.shape}")
print(f"Cropped center / Centro en recorte: x={x_center_crop:.2f}, y={y_center_crop:.2f}")


## 6. Visualizar la imagen observada / Visualize the observed image

Esta figura muestra cómo se ve el disco en el plano del cielo.  
This figure shows how the disk appears on the plane of the sky.


In [ ]:
def make_norm(img, percentile=99.5):
    interval = PercentileInterval(percentile)
    vmin, vmax = interval.get_limits(img)
    return ImageNormalize(vmin=vmin, vmax=vmax, stretch=AsinhStretch(0.1))

norm = make_norm(image_crop)

# Ejes con signo según el FITS / axes with FITS sign convention
x_axis_sky = (np.arange(image_crop.shape[1]) - x_center_crop) * cdelt_x
y_axis_sky = (np.arange(image_crop.shape[0]) - y_center_crop) * cdelt_y

extent_obs = [x_axis_sky[0], x_axis_sky[-1], y_axis_sky[0], y_axis_sky[-1]]

plt.figure(figsize=(7, 6))
plt.imshow(image_crop, origin="lower", cmap="inferno", norm=norm, extent=extent_obs)
plt.colorbar(label=header.get("BUNIT", "Intensity"))
plt.title(f"{DISK_NAME}: observed continuum image")
plt.xlabel("RA offset [arcsec]")
plt.ylabel("Dec offset [arcsec]")
plt.tight_layout()
plt.show()


## 7. Funciones geométricas / Geometry functions

La convención usada es:

The convention used is:

- `x_sky`: offset en RA / RA offset;
- `y_sky`: offset en Dec / Dec offset;
- `PA`: medido desde el norte hacia el este / measured east of north;
- `inclination`: inclinación del disco / disk inclination.

La transformación toma puntos del plano del disco y los proyecta al plano del cielo.  
The transformation takes points from the disk plane and projects them onto the sky plane.


In [ ]:
def disk_to_sky(x_disk, y_disk, inclination_deg, pa_deg):
    inc = np.deg2rad(inclination_deg)
    pa = np.deg2rad(pa_deg)

    # x_disk está sobre el eje mayor / x_disk lies along the major axis
    # y_disk está sobre el eje menor deproyectado / y_disk lies along the deprojected minor axis
    x_sky = x_disk * np.sin(pa) + (y_disk * np.cos(inc)) * np.cos(pa)
    y_sky = x_disk * np.cos(pa) - (y_disk * np.cos(inc)) * np.sin(pa)

    return x_sky, y_sky

def sky_to_pixel(x_sky, y_sky):
    x_pix = x_sky / cdelt_x + x_center_crop
    y_pix = y_sky / cdelt_y + y_center_crop
    return x_pix, y_pix

def sample_image_at_disk_coordinates(x_disk, y_disk):
    x_sky, y_sky = disk_to_sky(
        x_disk,
        y_disk,
        params["inclination_deg"],
        params["pa_deg"],
    )
    x_pix, y_pix = sky_to_pixel(x_sky, y_sky)

    return map_coordinates(
        image_crop,
        [y_pix, x_pix],
        order=1,
        mode="constant",
        cval=np.nan,
    )


## 8. Imagen deproyectada / Deprojected image

La imagen deproyectada intenta mostrar el disco como si se observara de frente.  
The deprojected image tries to show the disk as if it were seen face-on.

Si la geometría está bien aplicada, los anillos deberían verse más circulares que en la imagen observada.  
If the geometry is applied correctly, the rings should look more circular than in the observed image.


In [ ]:
r_max = params["r_max_arcsec"]
n_grid = 700

x_disk_axis = np.linspace(-r_max, r_max, n_grid)
y_disk_axis = np.linspace(-r_max, r_max, n_grid)
X_disk, Y_disk = np.meshgrid(x_disk_axis, y_disk_axis)

deproj_image = sample_image_at_disk_coordinates(X_disk, Y_disk)

plt.figure(figsize=(7, 6))
plt.imshow(
    deproj_image,
    origin="lower",
    cmap="inferno",
    norm=make_norm(deproj_image),
    extent=[-r_max, r_max, -r_max, r_max],
)
plt.colorbar(label=header.get("BUNIT", "Intensity"))
plt.title(f"{DISK_NAME}: deprojected image")
plt.xlabel("Disk-plane x [arcsec]")
plt.ylabel("Disk-plane y [arcsec]")
plt.tight_layout()
plt.show()


## 9. Mapa polar / Polar map

Ahora muestreamos la imagen directamente en coordenadas polares del plano del disco.  
Now we sample the image directly in polar coordinates of the disk plane.

Este método evita muchos artefactos de bordes que aparecen si primero se deproyecta una imagen rectangular.  
This method avoids many edge artifacts that appear when a rectangular image is deprojected first.


In [ ]:
n_r = 320
n_theta = 240

radii = np.linspace(0.0, r_max, n_r)
theta = np.linspace(-np.pi, np.pi, n_theta)

R, TH = np.meshgrid(radii, theta)

X_disk_polar = R * np.cos(TH)
Y_disk_polar = R * np.sin(TH)

polar_map = sample_image_at_disk_coordinates(X_disk_polar, Y_disk_polar)

plt.figure(figsize=(9, 5))
plt.imshow(
    polar_map,
    origin="lower",
    aspect="auto",
    cmap="inferno",
    norm=make_norm(polar_map),
    extent=[radii[0], radii[-1], -180, 180],
)
plt.colorbar(label=header.get("BUNIT", "Intensity"))
plt.title(f"{DISK_NAME}: polar map")
plt.xlabel("Radius [arcsec]")
plt.ylabel("Azimuthal angle [deg]")
plt.tight_layout()
plt.show()


## 10. Perfil radial / Radial profile

El perfil radial se obtiene promediando el mapa polar en el ángulo azimutal.  
The radial profile is obtained by averaging the polar map over azimuthal angle.

La banda sombreada muestra la desviación estándar azimutal.  
The shaded band shows the azimuthal standard deviation.


In [ ]:
radial_mean = np.nanmean(polar_map, axis=0)
radial_std = np.nanstd(polar_map, axis=0)

r_au = radii * params["distance_pc"]

plt.figure(figsize=(8, 5))
plt.plot(radii, radial_mean, lw=2, label="Mean intensity")
plt.fill_between(
    radii,
    radial_mean - radial_std,
    radial_mean + radial_std,
    alpha=0.25,
    label="Azimuthal standard deviation",
)
plt.xlabel("Radius [arcsec]")
plt.ylabel(header.get("BUNIT", "Intensity"))
plt.title(f"{DISK_NAME}: radial intensity profile")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(r_au, radial_mean, lw=2)
plt.xlabel("Radius [au]")
plt.ylabel(header.get("BUNIT", "Intensity"))
plt.title(f"{DISK_NAME}: radial intensity profile")
plt.tight_layout()
plt.show()


## 11. Candidatos a anillos y depresiones / Candidate rings and depressions

El algoritmo automático solo entrega candidatos.  
The automatic algorithm only provides candidates.

La decisión final debe revisarse visualmente usando el mapa polar y el perfil radial.  
The final decision should be visually checked using the polar map and the radial profile.


In [ ]:
def moving_average(y, window=9):
    kernel = np.ones(window) / window
    return np.convolve(y, kernel, mode="same")

profile_smooth = moving_average(radial_mean, window=11)

# Evitar la región central, donde el brillo puede dominar el perfil.
# Avoid the central region, where the brightness can dominate the profile.
mask = (radii > 0.08) & np.isfinite(profile_smooth)

valid_r = radii[mask]
valid_profile = profile_smooth[mask]

# Buscar máximos y mínimos locales con una separación mínima.
# Find local maxima and minima with a minimum separation.
peaks, _ = find_peaks(valid_profile, distance=14)
troughs, _ = find_peaks(-valid_profile, distance=14)

peak_r = valid_r[peaks]
trough_r = valid_r[troughs]

print("Candidate bright rings / Candidatos a anillos brillantes:")
for r in peak_r[:10]:
    print(f"  r = {r:.3f} arcsec ≈ {r * params['distance_pc']:.1f} au")

print("\nCandidate radial depressions / Candidatos a depresiones radiales:")
for r in trough_r[:10]:
    print(f"  r = {r:.3f} arcsec ≈ {r * params['distance_pc']:.1f} au")

plt.figure(figsize=(8, 5))
plt.plot(radii, radial_mean, alpha=0.45, label="Original profile")
plt.plot(radii, profile_smooth, lw=2, label="Smoothed profile")
plt.scatter(peak_r, np.interp(peak_r, radii, profile_smooth), marker="o", label="Candidate rings")
plt.scatter(trough_r, np.interp(trough_r, radii, profile_smooth), marker="v", label="Candidate depressions")
plt.xlabel("Radius [arcsec]")
plt.ylabel(header.get("BUNIT", "Intensity"))
plt.title(f"{DISK_NAME}: candidate substructures")
plt.legend()
plt.tight_layout()
plt.show()


## 12. Contraste de una brecha o depresión / Contrast of a gap or depression

Seleccionamos un radio de depresión y buscamos máximos cercanos a ambos lados.  
We select a depression radius and search for nearby maxima on both sides.

La fórmula usada es:

The formula used is:

$$
\delta_{\mathrm{gap}} = 1 - \frac{I_{\mathrm{gap}}}{I_{\mathrm{ring}}}
$$


In [ ]:
def classify_gap(delta):
    if delta < 0.3:
        return "weak depression / depresión débil"
    elif delta < 0.6:
        return "moderate gap or depression / brecha o depresión moderada"
    else:
        return "deep gap / brecha profunda"

def measure_gap_contrast(radii, profile, gap_radius, search_width=0.55, exclusion_width=0.04):
    # Intensidad en el mínimo seleccionado.
    # Intensity at the selected minimum.
    gap_idx = np.argmin(np.abs(radii - gap_radius))
    I_gap = profile[gap_idx]
    gap_radius = radii[gap_idx]

    # Buscar máximos cercanos a ambos lados.
    # Search for nearby maxima on both sides.
    inner_mask = (radii >= gap_radius - search_width) & (radii <= gap_radius - exclusion_width)
    outer_mask = (radii >= gap_radius + exclusion_width) & (radii <= gap_radius + search_width)

    if not np.any(inner_mask) or not np.any(outer_mask):
        raise RuntimeError(
            "No hay suficiente rango radial para medir el contraste. "
            "Elige una brecha más interna o aumenta search_width."
        )

    inner_profile = profile.copy()
    outer_profile = profile.copy()
    inner_profile[~inner_mask] = np.nan
    outer_profile[~outer_mask] = np.nan

    inner_idx = np.nanargmax(inner_profile)
    outer_idx = np.nanargmax(outer_profile)

    I_inner = profile[inner_idx]
    I_outer = profile[outer_idx]
    I_ring = 0.5 * (I_inner + I_outer)

    delta = 1.0 - I_gap / I_ring

    return {
        "gap_radius": gap_radius,
        "inner_ring_radius": radii[inner_idx],
        "outer_ring_radius": radii[outer_idx],
        "I_gap": I_gap,
        "I_inner": I_inner,
        "I_outer": I_outer,
        "I_ring": I_ring,
        "delta": delta,
    }

selected_gap = params["selected_gap_arcsec"]

result = measure_gap_contrast(
    radii,
    profile_smooth,
    selected_gap,
    search_width=0.60,
    exclusion_width=0.04,
)

print("Selected gap or depression / Brecha o depresión seleccionada")
print(f"Gap radius: {result['gap_radius']:.3f} arcsec ≈ {result['gap_radius'] * params['distance_pc']:.1f} au")
print(f"Inner ring radius: {result['inner_ring_radius']:.3f} arcsec ≈ {result['inner_ring_radius'] * params['distance_pc']:.1f} au")
print(f"Outer ring radius: {result['outer_ring_radius']:.3f} arcsec ≈ {result['outer_ring_radius'] * params['distance_pc']:.1f} au")
print()
print(f"I_gap = {result['I_gap']:.5e}")
print(f"I_ring_inner = {result['I_inner']:.5e}")
print(f"I_ring_outer = {result['I_outer']:.5e}")
print(f"I_ring = {result['I_ring']:.5e}")
print()
print(f"delta_gap = {result['delta']:.3f}")
print("Classification / Clasificación:", classify_gap(result["delta"]))

plt.figure(figsize=(8, 5))
plt.plot(radii, profile_smooth, lw=2, label="Smoothed radial profile")
plt.axvline(result["inner_ring_radius"], linestyle="--", label="Inner ring")
plt.axvline(result["gap_radius"], linestyle=":", label="Gap/depression")
plt.axvline(result["outer_ring_radius"], linestyle="--", label="Outer ring")
plt.xlabel("Radius [arcsec]")
plt.ylabel(header.get("BUNIT", "Intensity"))
plt.title(f"{DISK_NAME}: gap contrast measurement")
plt.legend()
plt.tight_layout()
plt.show()


## 13. Interpretación / Interpretation

Este análisis entrega una descripción morfológica del disco.  
This analysis provides a morphological description of the disk.

Para interpretar el resultado:

To interpret the result:

- revisa si los anillos aparecen como estructuras casi verticales en el mapa polar;
- check whether the rings appear as nearly vertical structures in the polar map;

- compara máximos y mínimos del perfil radial;
- compare maxima and minima in the radial profile;

- recuerda que el contraste medido es un contraste de brillo, no una depleción física directa;
- remember that the measured contrast is a brightness contrast, not a direct physical depletion;

- usa la imagen, el mapa polar y el perfil radial juntos, no una sola figura aislada.
- use the image, the polar map, and the radial profile together, not a single isolated figure.
